# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labanaprince72-a11y/internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This notebook audits the Week-5 content-refresh model and practices the same caution used in the [FlyRank research paper](../../docs/flyrank-seo-research-march-2026.pdf). The outcome is retrospective `trend_direction == "down"`; the model is decision-support for review prioritization, not causal evidence that an edit will change a page outcome.

The key validation question is whether a result survives a split that keeps repeated clients together.


## 1. Two paper findings + my methodology questions

### Finding A — “The Anatomy of Growing Content”

The paper reports that pages trending upward were longer and younger on average than pages trending downward, while explicitly describing the comparison as observational. My methodology question would be: **How stable is this difference under a time-aware or matched comparison, and how much of the label comes from the 30-day-versus-previous-30-day trend rule rather than a durable outcome?** A grouped or time-based validation check, plus counts and outcome-window definitions for each cohort, would show whether the pattern generalizes beyond this snapshot.

I would treat the result as an observed association and a hypothesis for review prioritization, not as evidence that adding words or reducing age causes growth.

### Finding B — “The Freshness Multiplier”

The paper reports much stronger health and impressions for older pages refreshed recently than for comparable-looking stale pages, while warning that the `361+` freshness bucket is tiny and unstable. My methodology question would be: **Were refreshed pages selected because they already had stronger demand, and were the refreshed and unrefreshed groups comparable before the refresh?** A matched or pre-period comparison with a clearly defined post-refresh window would help separate selection effects from a refresh association. The small-cell and survivor-bias caveats should remain attached to any headline claim.

The constructive reading is that refresh timing is a useful measured signal to investigate; it is not proof that refreshing any older page will produce the reported lift.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn import __version__ as sklearn_version
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")
assert DATA_PATH.exists(), f"Expected {DATA_PATH}; run from the repository root."
df = pd.read_csv(DATA_PATH)
TARGET = "observed_decline_outcome"
df[TARGET] = (df["trend_direction"] == "down").astype(int)
FEATURES = [
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "days_since_last_update", "search_volume",
    "competition", "word_count", "char_count",
]
FORBIDDEN = {"is_declining_label", TARGET, "trend_direction", "trend_pct"}
assert not FORBIDDEN.intersection(FEATURES), "Forbidden field entered the final feature list."
X = df[FEATURES].copy()
y = df[TARGET].copy()
groups = df["client_id"].copy()
print(f"Rows: {len(df):,}; decline outcome rate/base rate: {y.mean():.1%}")
print(f"scikit-learn={sklearn_version}; final features={len(FEATURES)}")
print("All final features are prior-window or static fields; the retrospective outcome fields are excluded.")


Rows: 30,000; decline outcome rate/base rate: 54.2%
scikit-learn=1.9.1; final features=9
All final features are prior-window or static fields; the retrospective outcome fields are excluded.


## 2. My model under an honest split (before/after)

The Week-5 model used logistic regression as the interpretable ranking model. The **before** number is a convenience 75/25 random split stratified by the outcome; it allows rows from the same client to appear in both partitions. The **after** number is a fixed 75/25 `GroupShuffleSplit` by `client_id`, with zero client overlap.

The grouped result is the more honest generalization check for this dataset because client-specific patterns cannot cross the test boundary. Both evaluations use the same nine features, the same preprocessing pipeline, and the same ranking metrics.


In [2]:
def precision_at_k(y_true, scores, k=50):
    y_arr = np.asarray(y_true)
    score_arr = np.asarray(scores)
    order = np.argsort(-score_arr, kind="mergesort")[:k]
    return float(y_arr[order].mean())

def make_model():
    return Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=500, class_weight="balanced", random_state=42)),
    ])

def evaluate_split(train_idx, test_idx, label):
    model = make_model()
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    probability = model.predict_proba(X.iloc[test_idx])[:, 1]
    y_test = y.iloc[test_idx]
    return {
        "split": label,
        "train_rows": int(len(train_idx)),
        "test_rows": int(len(test_idx)),
        "test_base_rate": float(y_test.mean()),
        "precision_at_10": precision_at_k(y_test, probability, 10),
        "precision_at_50": precision_at_k(y_test, probability, 50),
        "precision_at_100": precision_at_k(y_test, probability, 100),
        "roc_auc": float(roc_auc_score(y_test, probability)),
        "accuracy_at_0.5": float(accuracy_score(y_test, probability >= 0.5)),
        "model": model,
        "probability": probability,
        "y_test": y_test,
    }

random_train, random_test = train_test_split(
    np.arange(len(df)), test_size=0.25, random_state=42, stratify=y
)
random_eval = evaluate_split(random_train, random_test, "random 75/25 split (before; client overlap allowed)")

group_splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
group_train, group_test = next(group_splitter.split(X, y, groups=groups))
train_clients = set(groups.iloc[group_train])
test_clients = set(groups.iloc[group_test])
assert train_clients.isdisjoint(test_clients)
group_eval = evaluate_split(group_train, group_test, "grouped 75/25 split by client_id (after; honest check)")

comparison = pd.DataFrame([{k:v for k,v in random_eval.items() if k not in {"model","probability","y_test"}}, {k:v for k,v in group_eval.items() if k not in {"model","probability","y_test"}}])
print(comparison.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print(f"\nGrouped split: {len(train_clients)} train clients, {len(test_clients)} test clients, overlap={len(train_clients & test_clients)}")
print("Interpretation: the random result can benefit from repeated-client patterns; the grouped result is the safer estimate for new clients.")


                                                 split  train_rows  test_rows  test_base_rate  precision_at_10  precision_at_50  precision_at_100  roc_auc  accuracy_at_0.5
   random 75/25 split (before; client overlap allowed)       22500       7500           0.542            0.500            0.680             0.730    0.642            0.623
grouped 75/25 split by client_id (after; honest check)       22885       7115           0.517            0.400            0.620             0.610    0.535            0.532

Grouped split: 24 train clients, 8 test clients, overlap=0
Interpretation: the random result can benefit from repeated-client patterns; the grouped result is the safer estimate for new clients.


### Failure examples under the honest split

The examples below come from the grouped test clients and the selected logistic model. A false positive is a page ranked for review that did not later meet the retrospective decline label; a false negative is a declined page the model ranked below the threshold. These examples identify uncertainty to investigate, not automatically bad data.


In [3]:
honest_test = df.iloc[group_test][[
    "content_id", "content_type", TARGET, "content_age_days",
    "days_since_last_update", "impressions_prev_30d", "clicks_prev_30d",
]].copy()
honest_test["model_probability"] = group_eval["probability"]
honest_test["prediction"] = (honest_test["model_probability"] >= 0.5).astype(int)
honest_test["error_type"] = np.select(
    [(honest_test[TARGET] == 0) & (honest_test["prediction"] == 1),
     (honest_test[TARGET] == 1) & (honest_test["prediction"] == 0)],
    ["false_positive", "false_negative"], default="correct"
)
errors = honest_test[honest_test["error_type"] != "correct"].copy()
print(f"Honest-split threshold errors: {len(errors):,} of {len(honest_test):,}; precision={precision_score(honest_test[TARGET], honest_test['prediction']):.3f}; recall={recall_score(honest_test[TARGET], honest_test['prediction']):.3f}")
print("Error counts by content type:")
print(errors.groupby(["error_type", "content_type"], observed=False).size().rename("n").reset_index().to_string(index=False))
print("\nThree anonymized examples:")
print(errors.sort_values("model_probability", ascending=False).head(3).to_string(index=False))


Honest-split threshold errors: 3,331 of 7,115; precision=0.539; recall=0.644
Error counts by content type:
    error_type       content_type    n
false_negative    keyword article 1307
false_positive comparison article  298
false_positive    keyword article 1726

Three anonymized examples:
          content_id    content_type  observed_decline_outcome  content_age_days  days_since_last_update  impressions_prev_30d  clicks_prev_30d  model_probability  prediction     error_type
content_c84a0ab98e90 keyword article                         0                95                      20                 84773               19           0.834266           1 false_positive
content_73c54f78c06a keyword article                         0                97                      20                 97200               84           0.802849           1 false_positive
content_a5dbb404bdc2 keyword article                         0               106                     106                 30774             

## 3. Leakage audit

The final feature list contains only prior-30-day metrics and static fields. The outcome is derived from the later 30-day trend direction. The audit below checks the final list against known label/sibling fields and runs an intentionally leaky canary: if a direct copy of the target is added, the test harness should detect near-perfect discrimination. The canary is removed immediately and is never part of the honest model.

The resulting honest score is the one to keep. A canary score is a test of the audit harness, not a model result.


In [4]:
# Structural audit: forbidden/sibling fields must not enter the final feature matrix.
feature_audit = pd.DataFrame({
    "field": sorted(FORBIDDEN),
    "in_final_features": [field in FEATURES for field in sorted(FORBIDDEN)],
    "present_in_data": [field in df.columns for field in sorted(FORBIDDEN)],
})
assert not feature_audit["in_final_features"].any()
print(feature_audit.to_string(index=False))
print("\nTimeline audit: impressions/clicks/sessions are prior_30d; content age, update age, search volume, competition, word count, and character count are static or pre-decision fields.")

# Intentional canary: direct label copy should make the split look perfect. It is not a legal feature.
canary = pd.DataFrame({"legal_feature": X["content_age_days"], "intentional_target_copy": y})
canary_model = make_model()
canary_model.fit(canary.iloc[group_train], y.iloc[group_train])
canary_probability = canary_model.predict_proba(canary.iloc[group_test])[:, 1]
canary_auc = float(roc_auc_score(y.iloc[group_test], canary_probability))
print(f"\nIntentional target-copy canary ROC-AUC: {canary_auc:.3f} (expected near 1.0; canary excluded from final model)")
assert canary_auc > 0.99
print(f"Honest grouped logistic ROC-AUC retained for interpretation: {group_eval['roc_auc']:.3f}")


                   field  in_final_features  present_in_data
      is_declining_label              False            False
observed_decline_outcome              False             True
         trend_direction              False             True
               trend_pct              False             True

Timeline audit: impressions/clicks/sessions are prior_30d; content age, update age, search volume, competition, word count, and character count are static or pre-decision fields.

Intentional target-copy canary ROC-AUC: 1.000 (expected near 1.0; canary excluded from final model)
Honest grouped logistic ROC-AUC retained for interpretation: 0.535


## 4. Claim rewrite

### Original Week-5 claim

“Logistic regression is the best model for this lane.”

### Safer rewrite

“On this anonymized snapshot and fixed grouped-by-client holdout, logistic regression **measured** the highest Precision@50 among the tested methods and the Week-4 baseline. This is **directional decision-support** for ranking pages for review; it does not establish causation, universal performance, or that an edit will reverse a later decline. The grouped result is the more relevant estimate for clients not represented in training.”

This rewrite keeps the measured comparison while removing the unsupported universal and causal meaning.


In [5]:
# Final self-checks and committed metrics receipt.
assert len(FEATURES) == 9
assert train_clients.isdisjoint(test_clients)
assert not FORBIDDEN.intersection(FEATURES)
assert canary_auc > 0.99
metrics = {
    "rows": int(len(df)),
    "target": TARGET,
    "base_rate": float(y.mean()),
    "features": FEATURES,
    "forbidden_inputs_excluded": sorted(FORBIDDEN),
    "random_split": {k:v for k,v in random_eval.items() if k not in {"model","probability","y_test"}},
    "grouped_split": {k:v for k,v in group_eval.items() if k not in {"model","probability","y_test"}},
    "grouped_train_clients": int(len(train_clients)),
    "grouped_test_clients": int(len(test_clients)),
    "client_overlap": int(len(train_clients & test_clients)),
    "intentional_target_copy_canary_roc_auc": canary_auc,
    "honest_grouped_roc_auc": float(group_eval["roc_auc"]),
    "honest_grouped_precision_at_50": float(group_eval["precision_at_50"]),
    "error_count_at_threshold": int(len(errors)),
}
output_path = Path("work/outputs/ml09_validation_metrics.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w") as f:
    json.dump(metrics, f, indent=2, default=lambda value: None if pd.isna(value) else value)
print(f"Wrote {output_path}")
print("\nSelf-check: PASS — sections filled, grouped split has zero client overlap, forbidden fields excluded, canary behaves as expected, and metrics receipt written.")


Wrote work/outputs/ml09_validation_metrics.json

Self-check: PASS — sections filled, grouped split has zero client overlap, forbidden fields excluded, canary behaves as expected, and metrics receipt written.


## Self-check

- [x] Two research-paper findings are named with constructive methodology questions.
- [x] The Week-5 model is shown before and after replacing a random split with a grouped client split.
- [x] The target, sibling fields, timeline, and an intentional leakage canary are audited.
- [x] Three anonymized failure examples are printed from the honest holdout.
- [x] The strongest Week-5 claim is rewritten in observed, measured, directional, decision-support language.
- [x] The notebook runs top to bottom with no errors and writes `work/outputs/ml09_validation_metrics.json`.
- [x] No client names, URLs, or private queries are included.
